# Traffic Flow Prediction Dataset Generator
## Dhaka Urban Transport Network — GNN-ready Parquet Files

### Output (3 Parquet files, Snappy compressed)
```
tfp_edges_meta.parquet          — Static edge attributes WITH lat/lon coordinates
tfp_timestamps.parquet          — ts_idx → datetime/dow/hour/minute/is_weekend
tfp_traffic_timeseries.parquet  — (ts_idx, eidx, travel_time_s, current_speed_kmh, traffic_factor)
```

### Why a separate generator for Traffic Flow Prediction?
The routing generator (`dynamic_traffic_dataset_generator`) deliberately dropped
lat/lon and other spatial features because routing only needs `travel_time_s`.
Traffic Flow Prediction with GNNs needs:
- **Node/edge coordinates** (`u_lat`, `u_lon`, `v_lat`, `v_lon`, `mid_lat`, `mid_lon`)
  for spatial graph construction and positional encoding
- **`traffic_factor`** (congestion index 0–1) as a direct GNN prediction target
- **`current_speed_kmh`** as an alternative / interpretable prediction target
- **`haversine_m`** between connected nodes for distance-weighted adjacency

### Estimated output sizes
```
tfp_edges_meta.parquet          : ~3–5 MB    (N edges × 14 cols)
tfp_timestamps.parquet          : < 50 KB    (672 rows × 6 cols)
tfp_traffic_timeseries.parquet  : ~150–250 MB (N×672 rows × 5 cols)
```

In [6]:
# ===========================================
# 0) Install dependencies
# ===========================================
!pip install osmnx==1.9.3 geopandas shapely pyproj fiona networkx pyarrow
print("Installation complete.")

Installation complete.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# ===========================================
# 1) Imports
# ===========================================
import os, warnings, random, re, math
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import pyarrow as pa
import pyarrow.parquet as pq
from shapely.geometry import box
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

ox.settings.use_cache        = True
ox.settings.log_console      = False
ox.settings.requests_timeout = 300
ox.settings.cache_folder     = "./osmnx_cache"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("Imports successful.")

Imports successful.


In [8]:
# ===========================================
# 2) Parameters — adjust OUT_DIR to your path
# ===========================================
NETWORK_TYPE = "drive"
START        = "2025-01-06 00:00:00"   # Monday 00:00 (Asia/Dhaka)
END          = "2025-01-12 23:45:00"   # Sunday  23:45 -> exactly 7 days
FREQ         = "15min"                 # 96 steps/day -> 672 total

# ---- Change this to your output directory ----
OUT_DIR = r"D:\OneDrive\Desktop\AUST\Thesis_docs\traffic_flow_prediction"

# Bounding box for validation only (never used for graph filtering)
LAT_MIN, LAT_MAX = 23.60, 23.95
LON_MIN, LON_MAX = 90.25, 90.55

SAMPLE_EDGES       = None   # None = all edges; set e.g. 5000 to test quickly
MIN_SPEED_KPH      = 5.0
MIN_SPEED_FRACTION = 0.20
WEEKEND_DAYS       = {4, 5}   # Friday=4, Saturday=5 (BD public holiday)

BRTA_SPEED_LIMITS = {
    "motorway": 80, "trunk": 60, "primary": 50,
    "secondary": 40, "tertiary": 30, "unclassified": 25,
    "residential": 20, "living_street": 10, "service": 15,
    "motorway_link": 50, "trunk_link": 40, "primary_link": 35,
    "secondary_link": 30, "tertiary_link": 25,
}

# Output file paths
OUT_META   = os.path.join(OUT_DIR, "tfp_edges_meta.parquet")
OUT_TS_MAP = os.path.join(OUT_DIR, "tfp_timestamps.parquet")
OUT_TS     = os.path.join(OUT_DIR, "tfp_traffic_timeseries.parquet")

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output directory : {os.path.abspath(OUT_DIR)}")
print(f"Files will be written to:")
for f in [OUT_META, OUT_TS_MAP, OUT_TS]:
    print(f"  {f}")

Output directory : D:\OneDrive\Desktop\AUST\Thesis_docs\traffic_flow_prediction
Files will be written to:
  D:\OneDrive\Desktop\AUST\Thesis_docs\traffic_flow_prediction\tfp_edges_meta.parquet
  D:\OneDrive\Desktop\AUST\Thesis_docs\traffic_flow_prediction\tfp_timestamps.parquet
  D:\OneDrive\Desktop\AUST\Thesis_docs\traffic_flow_prediction\tfp_traffic_timeseries.parquet


In [9]:
# ===========================================
# 3) Acquire the Dhaka administrative polygon
# ===========================================
def get_admin_polygon(query, fallback_bbox=None):
    try:
        gdf  = ox.geocode_to_gdf(query)
        poly = gdf.geometry.unary_union
        print(f"Admin polygon CRS  : {gdf.crs}")
        print(f"Admin polygon type : {poly.geom_type}")
        print(f"Admin polygon bbox : {poly.bounds}")
        return poly
    except Exception as e:
        print(f"Nominatim query failed: {e}")
        if fallback_bbox:
            lat_min, lat_max, lon_min, lon_max = fallback_bbox
            print("Falling back to bounding-box polygon")
            return box(lon_min, lat_min, lon_max, lat_max)
        raise


def buffer_polygon_meters(poly, meters):
    gdf      = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")
    gdf_proj = gdf.to_crs("EPSG:32646")
    gdf_proj["geometry"] = gdf_proj.geometry.buffer(meters)
    return gdf_proj.to_crs("EPSG:4326").geometry.iloc[0]


PLACE_QUERY   = "Dhaka City Corporation, Dhaka, Bangladesh"
FALLBACK_BBOX = (LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)

admin_polygon          = get_admin_polygon(PLACE_QUERY, fallback_bbox=FALLBACK_BBOX)
admin_polygon_buffered = buffer_polygon_meters(admin_polygon, meters=200)

Nominatim query failed: Nominatim could not geocode query 'Dhaka City Corporation, Dhaka, Bangladesh' to a geometry of type (Multi)Polygon
Falling back to bounding-box polygon


In [10]:
# ===========================================
# 4) Download & prepare OSM road network
# ===========================================
print("Downloading OSM graph from admin polygon ...")
G_raw = ox.graph_from_polygon(
    admin_polygon_buffered,
    network_type=NETWORK_TYPE,
    simplify=True,
    retain_all=False,
    clean_periphery=True,
)
print(f"Raw: {G_raw.number_of_nodes():,} nodes, {G_raw.number_of_edges():,} edges")

print("Truncating to admin polygon ...")
G_truncated = ox.truncate.truncate_graph_polygon(
    G_raw,
    admin_polygon_buffered,
    retain_all=False,
    truncate_by_edge=True,
)
print(f"After truncation: {G_truncated.number_of_nodes():,} nodes, {G_truncated.number_of_edges():,} edges")

# Largest strongly connected component
G = ox.truncate.largest_component(G_truncated, strongly=True)
print(f"Largest SCC: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

Raw: 62,072 nodes, 156,756 edges
Truncating to admin polygon ...
After truncation: 62,072 nodes, 156,756 edges
Largest SCC: 61,937 nodes, 156,531 edges


In [11]:
# ===========================================
# 5) Add speeds and capacity factors
# ===========================================
# Remove maxspeed attribute (can cause TypeError in ox.add_edge_speeds)
for u, v, k, data in G.edges(keys=True, data=True):
    if "maxspeed" in data:
        del data["maxspeed"]

G = ox.add_edge_speeds(G, hwy_speeds=BRTA_SPEED_LIMITS)

# Normalize highway -> road_type
for u, v, k, data in G.edges(keys=True, data=True):
    hwy = data.get("highway")
    if isinstance(hwy, list):
        hwy = hwy[0]
    data["road_type"] = hwy if isinstance(hwy, str) else str(hwy)
    sp = data.get("speed_kph", None)
    try:
        data["free_speed_kph"] = float(sp) if sp is not None else np.nan
    except Exception:
        data["free_speed_kph"] = np.nan

# Fill NaN speeds from BRTA limits
for u, v, k, data in G.edges(keys=True, data=True):
    if not np.isfinite(data["free_speed_kph"]):
        data["free_speed_kph"] = float(BRTA_SPEED_LIMITS.get(data["road_type"], 25.0))

# Assign capacity factors (road capacity index [0,1])
CAPACITY_FACTORS = {
    "motorway": 0.85, "trunk": 0.80, "primary": 0.75,
    "secondary": 0.65, "tertiary": 0.55, "unclassified": 0.50,
    "residential": 0.45, "living_street": 0.35, "service": 0.40,
    "motorway_link": 0.70, "trunk_link": 0.65, "primary_link": 0.60,
    "secondary_link": 0.55, "tertiary_link": 0.50,
}
for u, v, k, data in G.edges(keys=True, data=True):
    rt = data.get("road_type", "unclassified")
    data["capacity_factor"] = CAPACITY_FACTORS.get(rt, 0.50)

print("Speeds and capacity factors assigned.")

Speeds and capacity factors assigned.


In [12]:
# ===========================================
# 6) Build edge_meta WITH lat/lon for GNN
# ===========================================
def haversine_m(lat1, lon1, lat2, lon2):
    """Haversine great-circle distance between two WGS-84 points (meters)."""
    R = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def build_tfp_edge_meta(G, sample=None, seed=42):
    """
    Build edge metadata DataFrame including spatial coordinates.

    Columns
    -------
    eidx            : integer index (joins traffic_timeseries)
    edge_id         : string 'u_v_k' (human-readable join key)
    u, v            : OSM node IDs
    u_lat, u_lon    : source node coordinates (WGS-84)
    v_lat, v_lon    : target node coordinates (WGS-84)
    mid_lat,mid_lon : edge midpoint coordinates
    haversine_m     : straight-line distance u->v in meters
    road_type       : OSM highway type
    length_m        : OSM edge length (meters)
    free_speed_kmh  : BRTA free-flow speed (km/h)
    capacity_factor : road-type capacity index [0, 1]
    """
    # Extract node coordinates once (OSM: y=lat, x=lon)
    node_coords = {
        nid: (data["y"], data["x"])
        for nid, data in G.nodes(data=True)
    }

    rows = []
    for u, v, k, d in G.edges(keys=True, data=True):
        u_lat, u_lon = node_coords.get(u, (np.nan, np.nan))
        v_lat, v_lon = node_coords.get(v, (np.nan, np.nan))
        mid_lat = (u_lat + v_lat) / 2.0
        mid_lon = (u_lon + v_lon) / 2.0
        h_dist = (haversine_m(u_lat, u_lon, v_lat, v_lon)
                  if all(np.isfinite([u_lat, u_lon, v_lat, v_lon])) else np.nan)

        rows.append({
            "edge_id":         f"{u}_{v}_{k}",
            "u":               int(u),
            "v":               int(v),
            "u_lat":           u_lat,
            "u_lon":           u_lon,
            "v_lat":           v_lat,
            "v_lon":           v_lon,
            "mid_lat":         mid_lat,
            "mid_lon":         mid_lon,
            "haversine_m":     h_dist,
            "road_type":       d.get("road_type", "unclassified"),
            "length_m":        float(d.get("length", np.nan)),
            "free_speed_kmh":  float(d.get("free_speed_kph", np.nan)),
            "capacity_factor": float(d.get("capacity_factor", 0.5)),
        })

    df = pd.DataFrame(rows)
    df.insert(0, "eidx", np.arange(len(df), dtype=np.uint32))

    if sample and sample < len(df):
        df = df.sample(sample, random_state=seed).reset_index(drop=True)
        df["eidx"] = np.arange(len(df), dtype=np.uint32)

    # Compact dtypes
    df["road_type"]       = df["road_type"].astype("category")
    df["length_m"]        = df["length_m"].round(1).astype(np.float32)
    df["free_speed_kmh"]  = df["free_speed_kmh"].round(2).astype(np.float32)
    df["capacity_factor"] = df["capacity_factor"].round(3).astype(np.float32)
    for col in ["u_lat", "u_lon", "v_lat", "v_lon", "mid_lat", "mid_lon", "haversine_m"]:
        df[col] = df[col].astype(np.float32)
    return df


edge_meta = build_tfp_edge_meta(G, sample=SAMPLE_EDGES)
n_edges   = len(edge_meta)
print(f"Edge metadata : {n_edges:,} edges x {len(edge_meta.columns)} columns")
print(f"Columns       : {list(edge_meta.columns)}")
display(edge_meta.head(3))

Edge metadata : 156,531 edges x 15 columns
Columns       : ['eidx', 'edge_id', 'u', 'v', 'u_lat', 'u_lon', 'v_lat', 'v_lon', 'mid_lat', 'mid_lon', 'haversine_m', 'road_type', 'length_m', 'free_speed_kmh', 'capacity_factor']


,eidx,edge_id,u,v,u_lat,u_lon,v_lat,v_lon,mid_lat,mid_lon,haversine_m,road_type,length_m,free_speed_kmh,capacity_factor
0,0,60917111_6139734068_0,60917111,6139734068,23.727516,90.420158,23.727055,90.420891,23.727285,90.420525,91.086182,primary,91.099998,50.0,0.75
1,1,60917111_387810798_0,60917111,387810798,23.727516,90.420158,23.727978,90.420448,23.727747,90.420303,59.390030,unclassified,59.400002,25.0,0.50
2,2,60917472_10016425303_0,60917472,10016425303,23.738119,90.395950,23.738068,90.395943,23.738092,90.395950,5.912888,secondary,5.900000,40.0,0.65


In [13]:
# ===========================================
# 7) Time index — 7-day window (Asia/Dhaka)
# ===========================================
tz         = "Asia/Dhaka"
t_start    = pd.Timestamp(START).tz_localize(tz)
t_end      = pd.Timestamp(END).tz_localize(tz)
time_index = pd.date_range(start=t_start, end=t_end, freq=FREQ)
n_ts       = len(time_index)

freq_min  = int(re.search(r"\d+", FREQ).group())
steps_day = (24 * 60) // freq_min
expected  = 7 * steps_day

assert n_ts == expected, (
    f"Expected {expected} timestamps, got {n_ts}. Check START/END/FREQ."
)
print(f"Time series : {n_ts:,} steps  (7 days x {steps_day}/day, dt={freq_min} min)")
print(f"Total rows  : {n_ts * n_edges:,}  ({n_ts * n_edges / 1e6:.1f}M)")

Time series : 672 steps  (7 days x 96/day, dt=15 min)
Total rows  : 105,188,832  (105.2M)


In [14]:
# ===========================================
# 8) Vectorized traffic model
# ===========================================
def base_time_of_day_factor(hour):
    """Baseline congestion index by hour — Dhaka-calibrated."""
    if hour < 5:  return 0.12
    if hour < 7:  return 0.28
    if hour < 10: return 0.82   # AM peak
    if hour < 12: return 0.38
    if hour < 14: return 0.48
    if hour < 17: return 0.40
    if hour < 20: return 0.85   # PM peak
    if hour < 22: return 0.35
    return 0.18


def traffic_factor_vec(ts, cap_arr):
    """
    Compute congestion factor for ALL edges at one timestamp (vectorized).
    Returns float32 array of shape (n_edges,) in range [0.05, 1.00].
    Higher value = more congested.
    """
    hour, dow = ts.hour, ts.dayofweek
    base      = base_time_of_day_factor(hour)

    if dow in WEEKEND_DAYS:
        base *= 0.60
        # Friday Jumu'ah mid-day spike
        if dow == 4 and 12 <= hour < 14:
            base *= 1.25

    # Lower capacity road -> higher congestion amplification
    factor = base * (1.0 + (1.0 - cap_arr) * 0.6)

    # Per-edge stochastic noise
    noise  = np.random.normal(0.0, 0.10, size=len(cap_arr)).astype(np.float32)
    return np.clip(factor + noise, 0.05, 1.00).astype(np.float32)


def compute_speed_and_tt(factor, free_kph_arr, length_m_arr):
    """From congestion factor -> current speed (km/h) and travel time (s)."""
    max_reduction = 1.0 - MIN_SPEED_FRACTION
    multiplier    = 1.0 - max_reduction * factor
    speed_kmh     = np.maximum(free_kph_arr * multiplier, MIN_SPEED_KPH).astype(np.float32)
    speed_mps     = speed_kmh * (1000.0 / 3600.0)
    tt_s          = (length_m_arr / speed_mps).astype(np.float32)
    return speed_kmh, tt_s


# Pre-extract numpy arrays (faster than per-row access)
_cap  = edge_meta["capacity_factor"].values.astype(np.float32)
_fkph = edge_meta["free_speed_kmh"].values.astype(np.float32)
_len  = edge_meta["length_m"].values.astype(np.float32)
_eidx = np.arange(n_edges, dtype=np.uint32)
print("Traffic model ready.")

Traffic model ready.


In [15]:
# ===========================================
# 9) Write File 1 — tfp_edges_meta.parquet
# ===========================================
edge_meta.to_parquet(OUT_META, compression="snappy", index=False)
meta_kb = os.path.getsize(OUT_META) / 1e3
print(f"tfp_edges_meta.parquet : {meta_kb:.1f} KB  ({n_edges:,} rows x {len(edge_meta.columns)} cols)")

tfp_edges_meta.parquet : 7514.9 KB  (156,531 rows x 15 cols)


In [16]:
# ===========================================
# 10) Write File 2 — tfp_timestamps.parquet
# ===========================================
ts_map = pd.DataFrame({
    "ts_idx":     np.arange(n_ts, dtype=np.uint16),
    "timestamp":  time_index,
    "dow":        time_index.dayofweek.astype(np.uint8),
    "hour":       time_index.hour.astype(np.uint8),
    "minute":     time_index.minute.astype(np.uint8),
    "is_weekend": time_index.dayofweek.isin(WEEKEND_DAYS).astype(np.uint8),
})
ts_map.to_parquet(OUT_TS_MAP, compression="snappy", index=False)
tsmap_kb = os.path.getsize(OUT_TS_MAP) / 1e3
print(f"tfp_timestamps.parquet : {tsmap_kb:.1f} KB  ({n_ts} rows x {len(ts_map.columns)} cols)")
display(ts_map.head(5))

tfp_timestamps.parquet : 12.7 KB  (672 rows x 6 cols)


,ts_idx,timestamp,dow,hour,minute,is_weekend
0,0,2025-01-06 00:00:00+06:00,0,0,0,0
1,1,2025-01-06 00:15:00+06:00,0,0,15,0
2,2,2025-01-06 00:30:00+06:00,0,0,30,0
3,3,2025-01-06 00:45:00+06:00,0,0,45,0
4,4,2025-01-06 01:00:00+06:00,0,1,0,0


In [17]:
# ===========================================
# 11) Write File 3 — tfp_traffic_timeseries.parquet
#     Columns: ts_idx, eidx, travel_time_s, current_speed_kmh, traffic_factor
# ===========================================
# Why 5 columns vs 3 in the routing generator:
#   traffic_factor    -> direct GNN prediction target (congestion index 0-1)
#   current_speed_kmh -> alternative prediction target (interpretable)
#   travel_time_s     -> kept for compatibility / combined feature use

schema = pa.schema([
    pa.field("ts_idx",            pa.uint16()),   # 0-671
    pa.field("eidx",              pa.uint32()),   # joins tfp_edges_meta.eidx
    pa.field("travel_time_s",     pa.float32()), # travel time in seconds
    pa.field("current_speed_kmh", pa.float32()), # speed in km/h
    pa.field("traffic_factor",    pa.float32()), # congestion index [0,1]
])

with pq.ParquetWriter(OUT_TS, schema, compression="snappy") as writer:
    for day in tqdm(range(7), desc="Writing traffic time-series", unit="day"):
        day_start = day * steps_day
        day_end   = day_start + steps_day

        ts_parts, tt_parts, sp_parts, tf_parts = [], [], [], []
        for ts_i in range(day_start, day_end):
            f              = traffic_factor_vec(time_index[ts_i], _cap)
            speed_kmh, tt  = compute_speed_and_tt(f, _fkph, _len)
            ts_parts.append(np.full(n_edges, ts_i, dtype=np.uint16))
            tt_parts.append(tt)
            sp_parts.append(speed_kmh)
            tf_parts.append(f)

        batch = pa.table({
            "ts_idx":            np.concatenate(ts_parts),
            "eidx":              np.tile(_eidx, steps_day),
            "travel_time_s":     np.concatenate(tt_parts),
            "current_speed_kmh": np.concatenate(sp_parts),
            "traffic_factor":    np.concatenate(tf_parts),
        }, schema=schema)
        writer.write_table(batch)

ts_mb = os.path.getsize(OUT_TS) / 1e6
print(f"tfp_traffic_timeseries.parquet : {ts_mb:.1f} MB  ({n_ts * n_edges:,} rows x 5 cols)")

Writing traffic time-series: 100%|██████████| 7/7 [00:41<00:00,  5.96s/day]

tfp_traffic_timeseries.parquet : 1564.6 MB  (105,188,832 rows x 5 cols)


In [18]:
# ===========================================
# 12) Summary & sanity check
# ===========================================
meta_kb  = os.path.getsize(OUT_META)   / 1e3
tsmap_kb = os.path.getsize(OUT_TS_MAP) / 1e3
ts_mb    = os.path.getsize(OUT_TS)     / 1e6

print("=" * 62)
print("  TFP Dataset Generation Complete")
print("=" * 62)
print(f"  tfp_edges_meta.parquet          : {meta_kb:>7.1f} KB   ({n_edges:,} edges x {len(edge_meta.columns)} cols)")
print(f"  tfp_timestamps.parquet          : {tsmap_kb:>7.1f} KB   ({n_ts} steps x {len(ts_map.columns)} cols)")
print(f"  tfp_traffic_timeseries.parquet  : {ts_mb:>7.1f} MB   ({n_ts * n_edges:,} rows x 5 cols)")
print(f"{'─'*62}")
print(f"  Total on disk                   : {ts_mb + (meta_kb + tsmap_kb)/1e3:.1f} MB")
print()
print("Join keys:")
print("  tfp_edges_meta.eidx   <->  tfp_traffic_timeseries.eidx")
print("  tfp_timestamps.ts_idx <->  tfp_traffic_timeseries.ts_idx")
print()
print("Columns available for GNN:")
print("  Spatial  : u_lat, u_lon, v_lat, v_lon, mid_lat, mid_lon, haversine_m")
print("  Road     : road_type, length_m, free_speed_kmh, capacity_factor")
print("  Temporal : dow, hour, minute, is_weekend  (via join on ts_idx)")
print("  Targets  : traffic_factor, current_speed_kmh, travel_time_s")
print()
print("Quick read-back sanity check:")
sample = pd.read_parquet(OUT_TS, engine="pyarrow").head(5)
display(sample)

  TFP Dataset Generation Complete
  tfp_edges_meta.parquet          :  7514.9 KB   (156,531 edges x 15 cols)
  tfp_timestamps.parquet          :    12.7 KB   (672 steps x 6 cols)
  tfp_traffic_timeseries.parquet  :  1564.6 MB   (105,188,832 rows x 5 cols)
──────────────────────────────────────────────────────────────
  Total on disk                   : 1572.1 MB

Join keys:
  tfp_edges_meta.eidx   <->  tfp_traffic_timeseries.eidx
  tfp_timestamps.ts_idx <->  tfp_traffic_timeseries.ts_idx

Columns available for GNN:
  Spatial  : u_lat, u_lon, v_lat, v_lon, mid_lat, mid_lon, haversine_m
  Road     : road_type, length_m, free_speed_kmh, capacity_factor
  Temporal : dow, hour, minute, is_weekend  (via join on ts_idx)
  Targets  : traffic_factor, current_speed_kmh, travel_time_s

Quick read-back sanity check:


,ts_idx,eidx,travel_time_s,current_speed_kmh,traffic_factor
0,0,0,7.717950,42.493145,0.187671
1,0,1,9.651331,22.156528,0.142174
2,0,2,0.638202,33.280998,0.209969
3,0,3,0.755219,46.238258,0.286703
4,0,4,0.665048,54.672737,0.110985
